In [1]:
"""
ACNE04: what happened when the same photograph was annotated twice?
===================================================================

The duplicate audit found 38 groups of byte-identical images (76 files).
Each group is an accident: the dataset holds one photograph under two
filenames. That accident is useful, because each group is a case where
ACNE04 annotated the SAME pixels twice.

This script asks one question of each group:

    did the two copies receive the same lesion count?

Two outcomes, and they mean opposite things:

  COUNTS IDENTICAL for most groups
      the annotation was copied along with the file. There is no second
      opinion here, only a second copy. The duplicates are then a leakage
      problem and nothing more - report them as such and stop.

  COUNTS DIFFER for many groups
      the same photograph was annotated independently twice, and the
      annotators disagreed. That is a direct, model-free measurement of
      annotation noise in the benchmark, and the 9 cross-grade pairs are
      the subset where that noise crossed a grade boundary.

Only the data can say which. Do not assume the second because it is the
more interesting one.

Run this AFTER verify_duplicates. See the header block for inputs.
"""

import csv
import json
import re
from collections import Counter, defaultdict
from pathlib import Path
from statistics import mean, median
import xml.etree.ElementTree as ET

In [2]:
# ---------------------------------------------------------------------------
# config
# ---------------------------------------------------------------------------

ROOT = "/kaggle/input/datasets/atkiyaanisa/acne04/acne04"
DUPLICATES_CSV = "fold_audit_out/duplicate_groups.csv"
OUT = "repeat_annotation_out"

CUTS = (5, 20, 50)          # Hayashi: <=5 mild, 6-20 moderate, 21-50 severe, >50
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def rule(count, cuts=CUTS):
    """Grade implied by a lesion count under the Hayashi criterion."""
    for i, c in enumerate(cuts):
        if count <= c:
            return i
    return len(cuts)

In [3]:
# ---------------------------------------------------------------------------
# locating inputs on Kaggle
# ---------------------------------------------------------------------------

def find_file(name, hint=None):
    """
    Audit CSVs are written to ./fold_audit_out/ inside THEIR OWN session.
    Attached to a new notebook they appear under /kaggle/input/<slug>/...
    instead, so search rather than trusting the relative path.
    """
    cands = []
    if hint:
        cands.append(Path(hint))
    cands += [Path(name), Path("fold_audit_out") / name]
    for c in cands:
        if c.is_file():
            return c
    for base in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if base.is_dir():
            try:
                hits = sorted(base.rglob(name))
            except OSError:
                hits = []
            if hits:
                print(f"  found {name} at {hits[0]}")
                return hits[0]
    return None

In [4]:
# ---------------------------------------------------------------------------
# reading the annotations
# ---------------------------------------------------------------------------

def parse_annotation_file(path):
    """Return [(image_name, [ints on that line])] for lines that name an image."""
    rows = []
    try:
        text = path.read_text(errors="replace")
    except Exception:
        return rows
    for line in text.splitlines():
        parts = line.replace(",", " ").split()
        if not parts:
            continue
        name = next((t for t in parts if Path(t).suffix.lower() in IMAGE_EXTS), None)
        if name is None:
            continue
        ints = []
        for tok in parts:
            if tok == name:
                continue
            try:
                ints.append(int(float(tok)))
            except ValueError:
                pass
        rows.append((Path(name).name, ints, path))
    return rows


N_GRADES = 4


def infer_columns(rows):
    """
    Decide which integer column is the grade and which is the lesion count.

    Reused verbatim from the week-1 audit, where it was validated against this
    dataset. Do NOT replace it with "the first integer on the line" - the
    classification lines carry the GRADE first, so that shortcut silently
    returns the grade and makes any count comparison circular.

      - the grade column takes few distinct values, all within 0..N_GRADES-1
      - the count column has a wider range and a higher max
    """
    widths = Counter(len(i) for _, i in rows)
    width = widths.most_common(1)[0][0]
    cols = [[] for _ in range(width)]
    for _, ints in rows:
        if len(ints) == width:
            for i, v in enumerate(ints):
                cols[i].append(v)

    stats = []
    for i, c in enumerate(cols):
        stats.append(None if not c else
                     {"index": i, "min": min(c), "max": max(c),
                      "n_distinct": len(set(c)), "mean": sum(c) / len(c)})

    grade_idx = None
    for s in stats:
        if s and s["min"] >= 0 and s["max"] <= N_GRADES - 1 and s["n_distinct"] <= N_GRADES:
            grade_idx = s["index"]
            break

    count_idx, best = None, -1
    for s in stats:
        if s and s["index"] != grade_idx and s["max"] > best:
            best, count_idx = s["max"], s["index"]

    return grade_idx, count_idx, {"line_widths": dict(widths), "column_stats": stats}


def collect_declared(root):
    """
    Declared lesion count per image, per annotation FILE.

    Classification lines carry 1-3 integers (grade, count); detection lines
    carry box coordinates and are excluded here.
    """
    files = sorted(p for p in root.rglob("*")
                   if p.suffix.lower() in {".txt", ".csv"}
                   and p.is_file() and p.stat().st_size > 0)
    all_rows, src_of = [], []
    for p in files:
        for name, ints, srcp in parse_annotation_file(p):
            if 1 <= len(ints) <= 3:
                all_rows.append((name, ints))
                src_of.append(srcp.name)

    if not all_rows:
        return {}, {}

    grade_idx, count_idx, infer = infer_columns(all_rows)
    print(f"    line widths seen      : {infer['line_widths']}")
    for st in infer["column_stats"]:
        if st:
            print(f"    column {st['index']}: min={st['min']} max={st['max']} "
                  f"distinct={st['n_distinct']} mean={st['mean']:.2f}")
    print(f"    inferred GRADE column : {grade_idx}")
    print(f"    inferred COUNT column : {count_idx}")
    if grade_idx is None or count_idx is None or grade_idx == count_idx:
        raise SystemExit(
            "could not separate grade from count - inspect the annotation lines "
            "before trusting any comparison")

    by_image = defaultdict(dict)
    label_grade = {}
    for (name, ints), srcname in zip(all_rows, src_of):
        if len(ints) > max(grade_idx, count_idx):
            by_image[name][srcname] = ints[count_idx]
            label_grade[name] = ints[grade_idx]
    return by_image, label_grade


def collect_boxes(root):
    """Number of annotated boxes per image, from VOC XML or flat box lines."""
    per_image = Counter()
    seen_xml = False
    for p in root.rglob("*.xml"):
        try:
            t = ET.parse(p)
            fn = t.findtext("filename") or (p.stem + ".jpg")
            per_image[Path(fn).name] = len(t.findall(".//object"))
            seen_xml = True
        except Exception:
            continue
    if seen_xml:
        return dict(per_image)
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() not in {".txt", ".csv"} or not p.is_file():
            continue
        for name, ints, _ in parse_annotation_file(p):
            if len(ints) >= 5:          # a box line, not a header line
                per_image[name] += 1
    return dict(per_image)


def load_groups(path):
    """{group_id: [image, ...]} plus the grade recorded for each image."""
    groups = defaultdict(list)
    grade = {}
    with open(path) as fh:
        for r in csv.DictReader(fh):
            if not r.get("image"):
                continue
            groups[r["group"]].append(r["image"])
            try:
                grade[r["image"]] = int(r["grade"])
            except (TypeError, ValueError):
                grade[r["image"]] = None
    return groups, grade

In [5]:
# ---------------------------------------------------------------------------
# the comparison
# ---------------------------------------------------------------------------

def one_count(declared, boxes, img):
    """
    Best single count for an image, and whether its sources agreed.

    Prefers the declared count (that is the dataset's own label); falls back
    to the box tally when no declared count exists.
    """
    d = declared.get(img, {})
    vals = sorted(set(d.values()))
    if vals:
        return vals[0], (len(vals) == 1), d, boxes.get(img)
    b = boxes.get(img)
    return b, True, {}, b


def main():
    root = Path(ROOT)
    out = Path(OUT)
    out.mkdir(parents=True, exist_ok=True)

    if not root.exists():
        raise SystemExit(f"dataset root not found: {root}")

    dup_path = find_file("duplicate_groups.csv", DUPLICATES_CSV)
    if dup_path is None:
        raise SystemExit(
            "duplicate_groups.csv not found. Run verify_duplicates first, then "
            "attach that notebook's output to this one.")

    groups, grade = load_groups(dup_path)
    print(f"  {len(groups)} duplicate groups, "
          f"{sum(len(v) for v in groups.values())} images")

    print("\n  reading annotations ...")
    declared, label_grade = collect_declared(root)
    boxes = collect_boxes(root)
    print(f"  declared counts for {len(declared)} images; "
          f"box tallies for {len(boxes)} images")

    flat = [v for d in declared.values() for v in d.values()]
    if flat and max(flat) <= N_GRADES - 1:
        raise SystemExit(
            "the 'count' column never exceeds %d, so it is almost certainly the "
            "GRADE, not the lesion count. Any comparison built on it would be "
            "circular. Fix the parser before reading anything below."
            % (N_GRADES - 1))
    print(f"  lesion counts range {min(flat)}-{max(flat)}, mean {mean(flat):.2f}")

    rows = []
    for gid, imgs in sorted(groups.items()):
        if len(imgs) < 2:
            continue
        info = []
        for im in sorted(imgs):
            c, consistent, srcs, nb = one_count(declared, boxes, im)
            info.append({"image": im, "count": c, "grade": grade.get(im),
                         "n_boxes": nb, "sources": srcs,
                         "self_consistent": consistent})
        # compare the first two members (every group here has exactly two)
        a, b = info[0], info[1]
        same_grade = (a["grade"] == b["grade"])
        if a["count"] is None or b["count"] is None:
            delta = None
        else:
            delta = abs(a["count"] - b["count"])
        rows.append({
            "group": gid,
            "image_a": a["image"], "image_b": b["image"],
            "grade_a": a["grade"], "grade_b": b["grade"],
            "count_a": a["count"], "count_b": b["count"],
            "count_delta": delta,
            "boxes_a": a["n_boxes"], "boxes_b": b["n_boxes"],
            "same_grade": same_grade,
            "counts_identical": (delta == 0) if delta is not None else None,
            "sources_a": ";".join(f"{k}={v}" for k, v in a["sources"].items()),
            "sources_b": ";".join(f"{k}={v}" for k, v in b["sources"].items()),
        })

    with (out / "repeat_annotation.csv").open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

    # -----------------------------------------------------------------
    report(rows, out)


def report(rows, out):
    usable = [r for r in rows if r["count_delta"] is not None]
    missing = len(rows) - len(usable)
    same = [r for r in usable if r["same_grade"]]
    cross = [r for r in usable if not r["same_grade"]]
    identical = [r for r in usable if r["count_delta"] == 0]
    differing = [r for r in usable if r["count_delta"] > 0]

    print("\n" + "=" * 74)
    print("SAME PHOTOGRAPH, ANNOTATED TWICE")
    print("=" * 74)
    print(f"  groups with counts on both copies : {len(usable)}"
          + (f"   ({missing} groups missing a count)" if missing else ""))
    print(f"  counts IDENTICAL                  : {len(identical)}")
    print(f"  counts DIFFER                     : {len(differing)}")
    if usable:
        print(f"  grade differs                     : {len(cross)} "
              f"({len(cross)/len(usable):.1%} of repeat annotations)")

    if differing:
        ds = [r["count_delta"] for r in differing]
        print(f"\n  among differing counts: mean |delta| {mean(ds):.2f}, "
              f"median {median(ds):.1f}, max {max(ds)}")

    print("\n" + "-" * 74)
    print("EVERY GROUP  (a = first filename, b = second)")
    print("-" * 74)
    print(f"  {'image a':>18} {'g':>2} {'cnt':>5}   "
          f"{'image b':<18} {'g':>2} {'cnt':>5}  {'d':>4}")
    for r in sorted(rows, key=lambda r: (r["same_grade"], r["group"])):
        d = "" if r["count_delta"] is None else str(r["count_delta"])
        mark = "  <- grade flip" if not r["same_grade"] else ""
        print(f"  {r['image_a']:>18} {str(r['grade_a']):>2} {str(r['count_a']):>5}   "
              f"{r['image_b']:<18} {str(r['grade_b']):>2} {str(r['count_b']):>5}  "
              f"{d:>4}{mark}")

    print("\n" + "=" * 74)
    print("WHAT THIS MEANS")
    print("=" * 74)
    if not usable:
        print("  No counts recovered for either copy. Check the annotation format")
        print("  before drawing any conclusion.")
    elif len(identical) == len(usable):
        print("  Every duplicate carries an IDENTICAL count. The annotation was")
        print("  copied along with the image file, so these are not second")
        print("  opinions and say nothing about annotator reliability.")
        print()
        print("  Report the duplicates as a LEAKAGE and integrity problem only.")
        print("  Do NOT describe them as repeat annotations or as evidence of")
        print("  label noise - that claim would not survive this table.")
        if cross:
            print()
            print(f"  The {len(cross)} cross-grade groups are then stranger, not weaker:")
            print("  the same image and the same count sit under two different grade")
            print("  folders, so the grade folder contradicts the count label.")
    elif len(differing) >= max(3, 0.2 * len(usable)):
        frac = len(differing) / len(usable)
        print(f"  {len(differing)} of {len(usable)} duplicate pairs ({frac:.0%}) received")
        print("  DIFFERENT lesion counts for identical pixels. These are genuine")
        print("  repeat annotations that the dataset made by accident.")
        print()
        print("  That is a direct, model-free measurement of annotation noise on")
        print("  this benchmark - no training run can be blamed for it, and no")
        print("  method can be credited with it.")
        print()
        if cross:
            print(f"  {len(cross)} of those disagreements crossed a grade boundary.")
            print("  Check whether the differing counts sit NEAR a cut-point "
                  f"{CUTS}:")
            print("  small disagreements landing on opposite sides of a cut is the")
            print("  boundary effect; large disagreements are a different problem.")
        print()
        print("  Caveat to state in the paper: the two copies may not have been")
        print("  annotated independently or blindly. Report what is observable -")
        print("  identical pixels, different labels - and do not claim a")
        print("  controlled inter-rater study.")
    else:
        print(f"  Mixed: {len(identical)} identical, {len(differing)} differing.")
        print("  Most annotation was copied, a minority was redone. Report both")
        print("  counts and treat only the differing pairs as repeat annotations.")

    print()
    print(f"  wrote {out/'repeat_annotation.csv'}")
    print("=" * 74)

    (out / "repeat_annotation.json").write_text(json.dumps({
        "n_groups": len(rows), "n_usable": len(usable),
        "n_counts_identical": len(identical), "n_counts_differ": len(differing),
        "n_grade_differs": len(cross), "cuts": list(CUTS),
    }, indent=2))


main()

  found duplicate_groups.csv at /kaggle/input/notebooks/atkiyaanisa/verifyduplicates-ipynb/fold_audit_out/duplicate_groups.csv
  38 duplicate groups, 76 images

  reading annotations ...
    line widths seen      : {2: 14570}
    column 0: min=0 max=3 distinct=4 mean=0.95
    column 1: min=1 max=65 distinct=56 mean=13.00
    inferred GRADE column : 0
    inferred COUNT column : 1
  declared counts for 1457 images; box tallies for 1457 images
  lesion counts range 1-65, mean 13.00

SAME PHOTOGRAPH, ANNOTATED TWICE
  groups with counts on both copies : 38
  counts IDENTICAL                  : 12
  counts DIFFER                     : 26
  grade differs                     : 9 (23.7% of repeat annotations)

  among differing counts: mean |delta| 4.42, median 2.5, max 16

--------------------------------------------------------------------------
EVERY GROUP  (a = first filename, b = second)
--------------------------------------------------------------------------
             image a  g   